# UrbanPulse QLoRA Training\n
\n
1. Upload `urbanpulse_multilang_qlora.jsonl` into `/content/`.\n
2. Run the cells below on a T4 GPU.\n
3. Download the LoRA adapter after training.

In [ ]:
!pip -q install unsloth transformers datasets peft trl accelerate bitsandbytes

In [ ]:
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer
from unsloth import FastLanguageModel

In [ ]:
MODEL_NAME = 'unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit'
DATASET_PATH = '/content/urbanpulse_multilang_qlora.jsonl'
OUTPUT_DIR = '/content/urbanpulse-qwen-lora'
MAX_SEQ_LENGTH = 1024

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],\n
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

In [ ]:
dataset = load_dataset('json', data_files=DATASET_PATH, split='train')

def format_sample(example):
    messages = example['messages']\n
    text = f"<|system|>\\n{messages[0]['content']}\\n<|user|>\\n{messages[1]['content']}\\n<|assistant|>\\n{messages[2]['content']}"
    return {'text': text}
dataset = dataset.map(format_sample, remove_columns=dataset.column_names)

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field='text',
    args=SFTConfig(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=2,
        learning_rate=2e-4,
        logging_steps=5,
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='linear',
        seed=42,
        report_to='none',
        max_seq_length=MAX_SEQ_LENGTH,
        save_strategy='epoch',
    ),
)
trainer.train()

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print('Saved adapter to', OUTPUT_DIR)